In [11]:
import gzip
from Bio import SeqIO
from collections import defaultdict

# === CONFIG ===
PFAM_MAX_LINES = 10000 # lines from Pfam mapping
SWISSPROT_MAX_RECORDS = 1000  # protein entries from FASTA
# ==============

base_path = "E:/Data/"
pfam_file_path = base_path + "Pfam-A.full.gz"
input_fasta = base_path + "uniprot_sprot.fasta.gz"

# STEP 1: Load partial Pfam-UniProt mappings
pfam_map = defaultdict(set)
print(f"Reading first {PFAM_MAX_LINES} lines from Pfam-A.full.uniprot.gz...")

with gzip.open(pfam_file_path, "rt") as pfam_file:
    for i, line in enumerate(pfam_file):
        print(line)
        if i >= PFAM_MAX_LINES:
            break
        # Skip comment lines
        if line.startswith("#") or not line.strip():
            continue
        parts = line.strip().split()
        print(parts)
        if len(parts) >= 2:
            uniprot_id, pfam_id = parts[0], parts[1]
            pfam_map[uniprot_id].add(pfam_id)

In [15]:
with gzip.open(pfam_file_path, "rt") as pfam_file:
    for i, line in enumerate(pfam_file):
        if i >= 10000:
            break
        print(line)

In [1]:
import gzip
from Bio import SeqIO

def pfam_block_iterator(filepath):
    """
    Generator that yields blocks from a Pfam-A.full.gz file.
    Each block ends with a line containing only '//'.
    """
    with gzip.open(filepath, 'rt') as f:
        block = []
        for line in f:
            block.append(line)
            if line.strip() == '//':
                yield ''.join(block)  # return as single string
                block = []

import re

def parse_pfam_block(block, max_gs_lines=10):
    """
    Extract:
    - Pfam domain ID from '#=GF ID'
    - First X #=GS line's accession and start-end
    Returns (pfam_id, accession, start, end) or None if not found
    """
    pfam_id = None
    acc = start = end = None

    # Extract Pfam domain ID
    for line in block.splitlines():
        if line.startswith("#=GF AC"):
            pfam_id = line.strip().split()[2]  # line: "#=GF ID PF00069"
            break

    res = []
    # Extract first x matching #=GS line
    counter = 0
    for line in block.splitlines():
        if line.startswith("#=GS") and "AC" in line:
            match = re.match(r"#=GS\s+(\S+)/(\d+)-(\d+)\s+AC\s+(\S+)", line)
            if match:
                full_acc = match.group(1)         # e.g., A0A7D5S632_UNCBA
                start = int(match.group(2))       # 1
                end = int(match.group(3))         # 78
                acc = full_acc.split('_')[0]      # Strip organism suffix
                res.append((pfam_id, acc, start, end))
                
                counter += 1
                if counter == max_gs_lines:
                    print()
                    break
    return res


from Bio.SeqRecord import SeqRecord

def find_sequence_by_accession(accession: str, records: list[SeqRecord]) -> str | None:
    """
    Search for the given accession in a list of SeqRecord objects and return the full sequence as a string.
    """
    for record in records:
        # Accession is usually in the format sp|P12345| or tr|...
        parts = record.id.split('|')
        if len(parts) >= 2:
            acc = parts[1]
        else:
            acc = record.id.split('.')[0]  # Fallback: strip version
        if acc == accession:
            return str(record.seq)
    return None

def write_graphpart_fasta(result_dict, output_path):
    """
    Writes the result_dict to a FASTA file in GraphPart-compatible format.
    """
    with open(output_path, 'w') as out_f:
        for i, (acc, (pfam_id, start, end, full_seq)) in enumerate(result_dict.items()):
            if full_seq is None:
                continue  # skip if sequence wasn't found
            header = f">{acc}|label={pfam_id}-{start}-{end}|priority={i}"
            out_f.write(f"{header}\n{full_seq}\n")
    print(f"✅ Wrote {len(result_dict)} entries to {output_path}")


# === Main script ===

pfam_file = "E:/Data/Pfam-A.full.gz"
swissprot_file = "E:/Data/uniprot_sprot.fasta.gz"
output_fasta = "data/pfam_subset_graphpart.fasta"

# Load Swiss-Prot records
with gzip.open(swissprot_file, 'rt') as f:
    records = list(SeqIO.parse(f, 'fasta'))

result_dict = {}

In [7]:

# Extract domain + sequence info
for i, block in enumerate(pfam_block_iterator(pfam_file)):
    print(f"Processing block {i+1}...")
    result = parse_pfam_block(block, max_gs_lines=40)
    for entry in result:
        pfam_id, acc, start, end = entry
        sequence = fetch_sequence(acc)
        if sequence:
            result_dict[acc] = (pfam_id, start, end, sequence)
    if i >= 400:
        break  # Limit to 200 families

# Write to formatted FASTA
write_graphpart_fasta(result_dict, "E:/Data/pfam_subset_graphpart_400x40.fasta")

In [3]:
import time
import requests

class RateLimitedSession:
    def __init__(self, max_requests_per_minute, max_retries=3, retry_delay=1):
        self.max_requests = max_requests_per_minute
        self.interval = 60.0 / max_requests_per_minute
        self.last_request_time = 0
        self.session = requests.Session()
        self.max_retries = max_retries
        self.retry_delay = retry_delay

    def _wait(self):
        now = time.time()
        elapsed = now - self.last_request_time
        wait_time = self.interval - elapsed
        if wait_time > 0:
            time.sleep(wait_time)
        self.last_request_time = time.time()

    def _request_with_retries(self, method, url, **kwargs):
        for attempt in range(self.max_retries):
            self._wait()
            response = self.session.request(method, url, **kwargs)

            if response.status_code == 200:
                return response
            else:
                print(f"[Attempt {attempt + 1}] Non-200 response: {response.status_code}")
                time.sleep(self.retry_delay)

        raise Exception(f"Failed to get a 200 response after {self.max_retries} attempts")

    def get(self, url, **kwargs):
        return self._request_with_retries('GET', url, **kwargs)

    def post(self, url, **kwargs):
        return self._request_with_retries('POST', url, **kwargs)
        
api = RateLimitedSession(max_requests_per_minute=200*60)

def fetch_sequence(protein_acc):
    url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{protein_acc}/"
    try:
        response = api.get(url)
        protein_info = response.json()
    except Exception as e:
        print(f"Error fetching data for {protein_acc}: {e}")
        return None
    sequence = protein_info["metadata"]["sequence"]
    return sequence

fetch_sequence("A0A0W7VNG5")

In [17]:
from Bio import SeqIO
import gzip

with gzip.open(input_fasta, "rt") as handle:
    records = list(SeqIO.parse(handle, "fasta"))

In [20]:
len(records)

In [19]:
import gzip
from Bio import SeqIO
from collections import defaultdict

def extract_pfam_accessions(pfam_file, max_entries):
    """
    Extract the first `max_entries` domain blocks and return a set of UniProt accessions.
    """
    accessions = dict()
    block_count = 0
    current_block = []

    with gzip.open(pfam_file, 'rt') as f:
        for line in f:
            current_block.append(line)
            if line.startswith('#=GS') and 'AC' in line:
                parts = line.strip().split()
                if len(parts) == 4:
                    accession = parts[3].split('.')[0]  # Strip version
                    accessions.add(accession)

            if line.strip() == '//':
                block_count += 1
                if block_count >= max_entries:
                    break
                current_block = []

    return accessions

def extract_sequences_from_swissprot(accessions, swissprot_fasta, output_fasta):
    """
    Write only the sequences whose accessions are in the given set.
    """
    count = 0
    with gzip.open(swissprot_fasta, 'rt') as in_f, open(output_fasta, 'w') as out_f:
        for record in SeqIO.parse(in_f, 'fasta'):
            acc = record.id.split('|')[1] if '|' in record.id else record.id.split('.')[0]
            if acc in accessions:
                SeqIO.write(record, out_f, 'fasta')
                count += 1
            if count >= len(accessions):
                break
    print(f"✅ Wrote {count} sequences to {output_fasta}")

# === Usage ===

base_path = "E:/Data/"
pfam_file = base_path + "Pfam-A.full.gz"
swissprot_file = base_path + "uniprot_sprot.fasta.gz"
output_fasta = base_path + "pfam_1000_sequences.fasta"

# Step 1: Get 1000 unique UniProt accessions from Pfam
accessions = extract_pfam_accessions(pfam_file, max_entries=1000)

# Step 2: Find and write their sequences from Swiss-Prot to a new FASTA
extract_sequences_from_swissprot(accessions, swissprot_file, output_fasta)


In [22]:
with gzip.open(swissprot_file, 'rt') as in_f:
    count = 0
    records = list(SeqIO.parse(in_f, 'fasta'))
        

In [24]:
records[1]